# 🎓 Full Fine-tuning (Tutorial-based / KcBERT)

공식 Smilegate UnSmile 튜토리얼 기반 Full Fine-tuning 노트북입니다.

## 📋 특징
- **모델**: `beomi/kcbert-base` (공식 튜토리얼 모델)
- **메트릭**: `LRAP` (Label Ranking Average Precision)
- **방식**: Full Fine-tuning (원본 튜토리얼과 동일)

## 🔗 참고
- [Smilegate AI UnSmile Tutorial](https://huggingface.co/smilegate-ai/kor_unsmile)

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import label_ranking_average_precision_score
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

## 2. 하이퍼파라미터 설정 (튜토리얼 기준)

In [ ]:
MODEL_NAME = "beomi/kcbert-base"
OUTPUT_DIR = "./output_tutorial_full"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 튜토리얼 하이퍼파라미터
EPOCHS = 5
BATCH_SIZE = 64  # 튜토리얼 기본값
LEARNING_RATE = 2e-5  # 튜토리얼 기본값
MAX_LENGTH = 128

LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령",
               "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
NUM_LABELS = len(LABEL_NAMES)

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

## 3. 데이터 로드

In [ ]:
TRAIN_PATH = "../3_UnSmile_Correction/unsmile_train_corrected.tsv"
VALID_PATH = "../3_UnSmile_Correction/unsmile_valid_corrected.tsv"

train_df = pd.read_csv(TRAIN_PATH, sep='\t', encoding='utf-8')
valid_df = pd.read_csv(VALID_PATH, sep='\t', encoding='utf-8')

print(f"Train 데이터: {len(train_df)}건")
print(f"Valid 데이터: {len(valid_df)}건")

## 4. 토크나이저 및 전처리

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    tokenized = tokenizer(
        examples['문장'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )
    labels = [[float(examples[col][i]) for col in LABEL_NAMES] 
              for i in range(len(examples['문장']))]
    tokenized['labels'] = labels
    return tokenized

train_dataset = Dataset.from_pandas(train_df).map(
    preprocess_function, batched=True, remove_columns=train_df.columns.tolist())
valid_dataset = Dataset.from_pandas(valid_df).map(
    preprocess_function, batched=True, remove_columns=valid_df.columns.tolist())

print("전처리 완료!")

## 5. 모델 로드

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)
model.config.id2label = {i: l for i, l in enumerate(LABEL_NAMES)}
model.config.label2id = {l: i for i, l in enumerate(LABEL_NAMES)}
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 6. 평가 메트릭 (튜토리얼: LRAP)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    return {'lrap': label_ranking_average_precision_score(labels, predictions)}

## 7. 학습 설정 및 실행

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="lrap",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

In [ ]:
print("학습 시작...")
trainer.train()
print("학습 완료!")

## 8. 모델 저장

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")
print(f"모델 저장 완료: {OUTPUT_DIR}/best_model")

## 9. 최종 평가

In [ ]:
print("="*60)
print("📊 최종 평가 결과 (Full FT Tutorial-based)")
print("="*60)

eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

with open(f"{OUTPUT_DIR}/results.txt", 'w', encoding='utf-8') as f:
    f.write("=== Full FT Tutorial-based (beomi/kcbert-base) ===\n")
    for k, v in eval_results.items():
        f.write(f"{k}: {v:.4f}\n")

print("="*60)
print("✅ 완료!")